[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/benchmarks/Benchmark_DependencyParsing.ipynb)

# Benchmark: Dependency Parsing

Scores a pretrained dependency parser's UAS (unlabeled attachment score) and LAS (labeled
attachment score) against gold-labeled data, using
`sparknlp.benchmark.Benchmark.evaluate(..., task="dependencyparsing")`.

**Dataset**: [Universal Dependencies English EWT](https://github.com/UniversalDependencies/UD_English-EWT),
test split (CC BY-SA 4.0) -- `HEAD` and `DEPREL` columns give the gold parse.

**Model**: `DependencyParserModel.pretrained()` + `TypedDependencyParserModel.pretrained()`,
Spark NLP's default pretrained English dependency parser pair (unlabeled parse, then labels).

## Setup

Run these cells first on a fresh Colab runtime.

In [3]:
!wget https://setup.johnsnowlabs.com/colab.sh -O - | bash

--2026-08-29 10:57:08--  https://setup.johnsnowlabs.com/colab.sh
Resolving setup.johnsnowlabs.com (setup.johnsnowlabs.com)... 3.86.22.73
Connecting to setup.johnsnowlabs.com (setup.johnsnowlabs.com)|3.86.22.73|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh [following]
--2026-08-29 10:57:08--  https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1483 (1.4K) [text/plain]
Saving to: ‘STDOUT’


-                     0%[                    ]       0  --.-KB/s               
-                   100%[===================>]   1.45K  --.-KB/s    in 0s      



In [4]:
# Current Colab runtimes default to Java 21, which Spark 3.4.x (what the bootstrap above
# installs) isn't compatible with -- Spark's low-level Platform.java reflection breaks on it.
# Switch to Java 17, which Spark 3.4.x does support.
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [5]:
# Current Colab runtimes also default to Python 3.13, which removed the deprecated
# `typing.io` submodule -- but Spark 3.4.x's own source still does `from typing.io import
# BinaryIO`. This patches that one import, both for this notebook process and for the
# separate Python worker subprocesses Spark launches to actually run distributed tasks
# (those load pyspark from its own bundled zip, so both copies need patching).
import zipfile, shutil

site_pkgs = "/usr/local/lib/python3.13/dist-packages"
loose_path = f"{site_pkgs}/pyspark/broadcast.py"
zip_path = f"{site_pkgs}/pyspark/python/lib/pyspark.zip"
OLD = "from typing.io import BinaryIO  # type: ignore[import]"
NEW = "from typing import BinaryIO  # patched for Python 3.13 (typing.io removed)"

with open(loose_path) as f:
    text = f.read()
with open(loose_path, "w") as f:
    f.write(text.replace(OLD, NEW))

tmp_path = zip_path + ".tmp"
with zipfile.ZipFile(zip_path, "r") as zin, zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        data = zin.read(item.filename)
        if item.filename == "pyspark/broadcast.py":
            data = data.decode("utf-8").replace(OLD, NEW).encode("utf-8")
        zout.writestr(item, data)
shutil.move(tmp_path, zip_path)
print("Environment patched for this Colab runtime (Java 17, typing.io).")

Environment patched for this Colab runtime (Java 17, typing.io).

In [6]:
import sparknlp
spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)
from sparknlp.training import CoNLL
from sparknlp.annotator import DependencyParserModel, TypedDependencyParserModel
from pyspark.ml import Pipeline
from pyspark.sql.functions import expr, concat_ws
from sparknlp.benchmark import Benchmark

Spark NLP version: 6.4.2
Apache Spark version: 3.4.4

## 1. Get some data

In [8]:
import urllib.request

url = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-test.conllu"
conllu_text = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")

sentences = []
current = []
for line in conllu_text.split("\n"):
    if line.startswith("#"):
        continue
    if line.strip() == "":
        if current:
            sentences.append(current)
            current = []
        continue
    cols = line.split("\t")
    if "-" in cols[0] or "." in cols[0]:
        continue  # multiword-token / empty-node lines
    # (FORM, XPOS, HEAD (1-indexed, 0=root), DEPREL)
    current.append((cols[1], cols[4], int(cols[6]), cols[7]))
if current:
    sentences.append(current)

print(len(sentences), "sentences")
print(sentences[0])

2077 sentences
[('What', 'WP', 0, 'root'), ('if', 'IN', 4, 'mark'), ('Google', 'NNP', 4, 'nsubj'), ('Morphed', 'VBD', 1, 'advcl'), ('Into', 'IN', 6, 'case'), ('GoogleOS', 'NNP', 4, 'obl'), ('?', '.', 4, 'punct')]

Same CoNLL-4-column trick as the POS notebook, to get a `token`/`pos` column that's
guaranteed aligned with our gold tokenization. The gold head/label pairs are attached with a
**join on the reconstructed token sequence** rather than by row position -- Spark DataFrames
don't guarantee row order survives a read, so zipping a Python list back on by position would
risk silently misaligning some sentences.

A handful of short sentences (e.g. "Thanks !") share an identical token sequence, though, so a
plain join on that sequence isn't safe either -- it would silently fan out and pair gold deps
with the wrong sentence. We only keep sentences whose token sequence is unique on both sides
before joining, and drop the (small number of) ambiguous duplicates instead of guessing.

In [10]:
conll_path = "/tmp/ud_english_ewt_test_dep.conll2003"
with open(conll_path, "w") as f:
    for sent in sentences:
        for form, xpos, head, deprel in sent:
            f.write(f"{form} {xpos} O O\n")
        f.write("\n")

gold_data = CoNLL().readDataset(spark, conll_path)
gold_data = gold_data.withColumn(
    "token_key", concat_ws(" ", expr("transform(token, x -> x.result)")))

dep_rows = [
    (" ".join(form for form, _, _, _ in sent),
     [f"{head}:{deprel}" for _, _, head, deprel in sent])
    for sent in sentences
]
dep_lookup = spark.createDataFrame(dep_rows, ["token_key", "gold_deps"])

unique_gold_keys = gold_data.groupBy("token_key").count().filter("count = 1").select("token_key")
gold_data = gold_data.join(unique_gold_keys, on="token_key", how="inner")

unique_dep_keys = dep_lookup.groupBy("token_key").count().filter("count = 1").select("token_key")
dep_lookup = dep_lookup.join(unique_dep_keys, on="token_key", how="inner")

gold_data = gold_data.join(dep_lookup, on="token_key", how="inner")
print(gold_data.count(), "sentences matched (of", len(sentences), "parsed)")
gold_data.select("text", "gold_deps").show(2, truncate=80)

1915 sentences matched (of 2077 parsed)
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                            text|                                                                       gold_deps|
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|" ... there is no companion quite so devoted , so communicative , so loving a...|[4:punct, 4:punct, 4:expl, 0:root, 6:det, 4:nsubj, 9:advmod, 9:advmod, 6:amod...|
|     " ... to gaze at Wei 's art is like entering a floating world of dreams . "|[11:punct, 11:punct, 4:mark, 11:csubj:outer, 8:case, 8:nmod:poss, 6:case, 4:o...|
+--------------------------------------------------------------------------------+----------------------------------------------------------

## 2. Build the pipeline

In [12]:
dep_parser = DependencyParserModel.pretrained() \
    .setInputCols(["sentence", "pos", "token"]).setOutputCol("dependency")
typed_dep_parser = TypedDependencyParserModel.pretrained() \
    .setInputCols(["token", "pos", "dependency"]).setOutputCol("labeled_dependency")

pipeline = Pipeline(stages=[dep_parser, typed_dep_parser])
pipeline_model = pipeline.fit(gold_data)

dependency_conllu download started this may take some time.
Approximate size to download 16.7 MB

[ | ]
[OK!]
dependency_typed_conllu download started this may take some time.
Approximate size to download 2.4 MB

[ | ]
[OK!]

> **Note: check begin/end and index conventions before trusting the score.** Before
> running the full benchmark, it's worth confirming Spark NLP's head-index convention actually
> matches the gold encoding above (1-indexed token position, `0` = root) -- a silent convention
> mismatch (e.g. a different root marker) would make every score look wrong even for a good
> model. On a quick single-sentence check, `dependency.metadata['head']` matched the gold `HEAD`
> column exactly, confirming the encoding lines up.

## 3. Run the benchmark

In [15]:
report = Benchmark.evaluate(pipeline_model, gold_data, task="dependencyparsing", label_col="gold_deps")
print(report)

dependencyparsing accuracy (n=24442): las=0.2771, uas=0.7025

## Reading the result

A LAS noticeably lower than UAS is common and often reflects differences between the parser's
own dependency-label inventory and the gold treebank's, on top of genuine attachment errors --
not necessarily a scorer problem.